# Iron man 2 — the v3.4 version (VEi) against a correctly built BC

**VEi** (the locked v3.4 version: small-canvas reference → SR → fal-canvas edit, no ankle cut) against **BC** (v3.1's incumbent, rebuilt properly: bald pass → head subtracted by the V2 cropper → edit), 200-pair matrix, seeds **46/47/48**. ~1,312 klein calls. **Self-contained**: the only image inputs are the raw person/garment photographs in the bundle's `testset/`; normalise, BiRefNet A4 crops, framing reads, and every klein call run fresh on this A100 (Drive supplies only model-weight caches).

**Two sessions, because the V2 head-subtraction cropper runs locally:**
- **Session 1 — Run all.** VEi refs + 600 VEi edits + 56 BC bald frames → zip to Drive. Then locally: `python3 v3/build/ironman_bc_crop.py <unzipped run dir>` and upload `v34_bc_refs.zip` (the `refs/*__BC.jpg`) to Drive `v3_runs/`.
- **Session 2 — run cells 1–4, then 7–8** (skip 5–6). Cell 7 pulls the BC refs from Drive and runs the 600 BC edits at the fal call-2 canvas (`bc_canvas='fal'`).

Open directly: https://colab.research.google.com/github/101011101/magichour_takehome/blob/v3.3-lock/v3/colab/v34_a100.ipynb — Runtime → A100.


In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 0.689     # CAD/h at 5.3 CU/h x CAD 0.13/CU; edit if your rate differs
SEEDS = [46, 47, 48]          # the iron-man seeds: comparable with the v3.3 record
ARMS = ("VEi",)               # the locked v3.4 version; BC handled per stage below
MATRICES = ["matrix.csv"]     # the 200-pair iron-man matrix
DRIVE_PROJECT_DIR = "Side projects and shi"

In [ ]:
# 2 · Drive: find the HF cache that holds klein; reuse the iron-man run
import os
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'; BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
candidates = [os.path.join(MYDRIVE, 'hf_cache'), os.path.join(BASE, 'tryon_models', 'hf_cache'), os.path.join(BASE, 'hf_cache')]
found = [c for c in candidates if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
os.environ['HF_HOME'] = found[0] if found else candidates[0]
os.environ['V3_MODEL_DIR'] = os.path.join(BASE if os.path.isdir(BASE) else MYDRIVE, 'v3_models')
os.makedirs(os.environ['HF_HOME'], exist_ok=True); os.makedirs(os.environ['V3_MODEL_DIR'], exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'], '(klein cached)' if found else '(no cached klein - cell 4 downloads ~13 GB once)')


In [ ]:
# 3 · install; pull the bundle from GitHub (public, branch v3.3-lock)
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe onnxruntime-gpu opencv-python-headless
!cd /content && rm -rf v34 && wget -q -O v33_ironman_bundle.zip https://github.com/101011101/magichour_takehome/raw/v3.3-lock/v33_ironman_bundle.zip && unzip -qo v33_ironman_bundle.zip -d v34
%cd /content/v34
import os, zipfile, onnxruntime as ort, torch
assert os.path.exists('realesr-general-x4v3.pth') and os.path.exists('lib/run_ironman.py') and os.path.exists('v34_failures.csv') and os.path.exists('v34_controls.csv'), 'bundle incomplete'
os.makedirs('run/inputs', exist_ok=True)   # self-contained: no previous run reused - normalise and A4 crops computed fresh from testset/ (raw photos in the bundle)
print('onnxruntime providers:', ort.get_available_providers(), '| gpu:', torch.cuda.get_device_name(0))

In [ ]:
# 4 · load klein once, timed
import sys; sys.path.insert(0, 'lib')
import klein_local as K
K.load(); K.info()

In [ ]:
# 5 · one pair first (session 1)
import run_ironman as R
R.main(MATRICES[0], 'testset', limit=1, seeds=SEEDS[:1], arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
print(sorted(f for f in os.listdir('run/gen')))


In [ ]:
# 6 · session 1: the full VEi arm, then BC's bald frames (resumable)
R.main(MATRICES[0], 'testset', limit=None, seeds=SEEDS, arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
R.main(MATRICES[0], 'testset', limit=None, seeds=SEEDS, arms=("BC",), gpu_usd_per_hour=A100_USD_PER_HOUR, stage="bald")
import json; print(json.dumps(json.load(open('run/meta/cost.json')), indent=1))


In [ ]:
# 7 · session 2: BC edits from the locally-made head-subtracted refs
import glob, zipfile as zf
import run_ironman as R
bc = sorted(glob.glob(os.path.join(BASE, 'v3_runs', 'v34_bc_refs*.zip')))
if not bc:
    print('session 2 skipped: no v34_bc_refs*.zip on Drive - run ironman_bc_crop.py locally first')
else:
    for f in glob.glob('run/refs/*__BC.jpg'): os.remove(f)   # purge the PREV zip's head-ful BCA4-style refs
    names = [n for n in zf.ZipFile(bc[-1]).namelist() if n.endswith('__BC.jpg')]
    zf.ZipFile(bc[-1]).extractall('run')   # refs/{g}__BC.jpg
    n = len([f for f in os.listdir('run/refs') if f.endswith('__BC.jpg')])
    assert n == len(names) and n >= 56, f'{n} BC refs on disk vs {len(names)} in the zip'
    print(f'{n} head-subtracted BC refs from', os.path.basename(bc[-1]))
    R.main(MATRICES[0], 'testset', limit=None, seeds=SEEDS, arms=("BC",), gpu_usd_per_hour=A100_USD_PER_HOUR, stage="bcedit", bc_canvas="fal")


In [ ]:
# 7 · zip this run's references, outputs and meta to Drive
import shutil, time
name = f"v34_ironman2_{time.strftime('%Y%m%d_%H%M')}"
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('run/refs'):
        if any(a in f for a in ('__VEi', '__bald', '__BC', '__VS', '__VA', '__VE', '__V34', '__Vnc', '__Vfc')): z.write('run/refs/' + f, 'refs/' + f)
    for f in os.listdir('run/gen'): z.write('run/gen/' + f, 'gen/' + f)
    for f in os.listdir('run/inputs'): z.write('run/inputs/' + f, 'inputs/' + f)
    for f in os.listdir('run/meta'): z.write('run/meta/' + f, 'meta/' + f)
os.makedirs(os.path.join(BASE, 'v3_runs'), exist_ok=True); shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', name + '.zip'))
print('->', os.path.join(BASE, 'v3_runs', name + '.zip'), ' then locally: python3 v3/build/v34_a100_page.py <unpacked dir> --arm V34')